In [ ]:
# -*- coding: utf-8 -*-
# 전국도시가스 용도별 수요가수/공급량 정리 (2001~현재)
# - data 폴더의 모든 엑셀 처리
# - 블록폭:
#   2001~2004: 수요 13, 공급 15  => 2+13+15=30열
#   2005~2018: 수요 14, 공급 16  => 2+14+16=32열
#   2019~현재: 수요 16, 공급 18  => 2+16+18=36열
# - 2016~현재: '부피' 시트만 처리(이름 정규화 후 탐색)
# - 열병합용1+2 → 열병합용, 열전용설비용/집단용 → 집단에너지용(연속성 확보)
# - 최종 long: [연도, 시도, 회사, 용도, 수요가수, 공급량]

from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
from openpyxl import load_workbook

# ================== 경로 ==================
BASE = Path(r"D:\Project\전국도시가스용도별수요가수공급량")
DATA = BASE / "data"
OUT  = BASE / "out"
OUT.mkdir(parents=True, exist_ok=True)

# ✅ 임시파일 폴더
TMP_DIR = BASE / "Temp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

# ================== 유틸 ==================
def unmerge_save(src: Path, dst: Path):
    wb = load_workbook(src, data_only=True)
    for ws in wb.worksheets:
        for r in list(ws.merged_cells.ranges):
            ws.unmerge_cells(str(r))
    wb.save(dst)

def as_series(df: pd.DataFrame, col: str) -> pd.Series:
    obj = df.loc[:, col]
    if isinstance(obj, pd.DataFrame):
        obj = obj.iloc[:, 0]
    return obj

def set_series(df: pd.DataFrame, col: str, s: pd.Series):
    if (df.columns == col).sum() >= 2:
        first_idx = list(df.columns).index(col)
        df.iloc[:, first_idx] = s.values
    else:
        df[col] = s

def numify(df: pd.DataFrame, cols):
    for c in cols:
        s = as_series(df, c).astype(str)
        s = (
            s.str.replace(",", "", regex=False)
             .str.replace(r"[^\d.\-]", "", regex=True)
             .replace({"": None})
        )
        set_series(df, c, pd.to_numeric(s, errors="coerce"))
    return df

def clean_fill_idcols(df: pd.DataFrame):
    for c in ("시도","회사"):
        s = as_series(df, c).astype(str)
        s = (
            s.str.replace(r"\s+", "", regex=True)
             .replace({"nan": None})
             .ffill()
        )
        set_series(df, c, s)
    return df

def drop_noise_rows(df: pd.DataFrame):
    patt = r"(계|구성비)|(^\s*주\s*\)?)|(^\s*\d+\.)"
    s_sido = as_series(df, "시도").fillna("").astype(str)
    s_comp = as_series(df, "회사").fillna("").astype(str)
    mask = (
        s_sido.str.contains(patt, regex=True) |
        s_comp.str.contains(patt, regex=True) |
        (s_sido.str.strip().eq("") & s_comp.str.strip().eq(""))
    )
    return df.loc[~mask].copy()

def drop_mostly_text_cols(df, id_cols=("시도","회사"), keep_ratio=0.2):
    non_id = [c for c in df.columns if c not in id_cols]
    bad = []
    for c in non_id:
        s = df[c].astype(str)
        sclean = (
            s.str.replace(",", "", regex=False)
             .str.replace(r"[^\d.\-]", "", regex=True)
        )
        numeric_rate = pd.to_numeric(sclean, errors="coerce").notna().mean()
        if numeric_rate < keep_ratio:
            bad.append(c)
    return df.drop(columns=bad, errors="ignore")

# ============ 2016+ 파일: '부피' 시트 선택 ============

def _norm_sheet(s: str) -> str:
    s = unicodedata.normalize("NFKC", s).lower()
    s = re.sub(r"[\u200b-\u200d\u2060\uFEFF\u00a0\u202f]", "", s)  # 제로폭/특수공백 제거
    s = re.sub(r"\s+|[\-_/·•∙ㆍ]", "", s)
    s = re.sub(r"[()\[\]{}]", "", s)
    return s

def pick_volume_sheet(xf: pd.ExcelFile) -> str:
    names = xf.sheet_names
    norm_map = {nm: _norm_sheet(nm) for nm in names}
    for nm, nn in norm_map.items():
        if "부피" in nn:
            return nm
    for k in ("부피량","m3","천m3","volume"):
        for nm, nn in norm_map.items():
            if k in nn:
                return nm
    # 마지막 안전망: 첫 시트
    print(f"[알림] '부피' 시트를 찾지 못했습니다. 첫 시트 사용: {names[0]}")
    return names[0]

# ================== 블록 폭 규칙 ==================
def get_block_widths(year: int) -> tuple[int, int]:
    if year <= 2004:
        return 13, 15
    elif year <= 2018:
        return 14, 16
    else:  # 2019+
        return 16, 18

# ================== 라벨 템플릿 ==================
def make_demand_labels(width: int, year: int) -> list:
    """
    수요가수(블록 왼쪽) 헤더 라벨
    - 2001~2004: 13열
    - 2005~2018: 14열
    - 2019~현재: 16열
    """
    labels_2001_2004 = [
        "가정용","난방용","일반용1","일반용2","일반용소계",
        "업무용","냉난방용","산업용","열병합용","수송용",
        "합계","증감률","구성비",
    ]  # 13

    labels_2005_2018 = [
        "가정용","난방용","일반용1","일반용2","일반용소계",
        "업무용","냉난방용","산업용","열병합용","집단용","수송용",
        "합계","증감률","구성비",
    ]  # 14

    labels_2019_plus = [
        "가정용","난방용","일반용1","일반용2","일반용소계",
        "업무용","냉난방용","산업용",
        "열병합용1","열병합용2","열전용설비용",
        "수송용","연료전지용",
        "합계","증감률","구성비",
    ]  # 16

    if year <= 2004:
        base = labels_2001_2004
    elif year <= 2018:
        base = labels_2005_2018
    else:
        base = labels_2019_plus

    # 안전장치: 정의된 길이와 실제 width가 다를 때 슬라이스 / 패딩
    if width == len(base):
        return base
    elif width < len(base):
        # 엑셀 열이 더 적으면 앞에서부터 잘라서 사용
        return base[:width]
    else:
        # 엑셀 열이 더 많으면 남는 부분은 generic 이름으로 채움
        extra = [f"col{i+1}" for i in range(width - len(base))]
        return base + extra

def supply_labels(year: int, width: int) -> list:
    """
    공급량(블록 오른쪽) 헤더 라벨
    - 2001~2004: 15열
    - 2005~2018: 16열
    - 2019~현재: 18열
    """
    labels_2001_2004 = [
        "취사용","개별난방용","가정용소계",
        "일반용1","일반용2","일반용소계",
        "업무난방용","냉난방용","업무용소계",
        "산업용","열병합용","수송용",
        "합계","증감률","구성비",
    ]  # 15

    labels_2005_2018 = [
        "취사용","개별난방용","가정용소계",
        "일반용1","일반용2","일반용소계",
        "업무난방용","냉난방용","업무용소계",
        "산업용","열병합용","집단용","수송용",
        "합계","증감률","구성비",
    ]  # 16

    labels_2019_plus = [
        "취사용","개별난방용","가정용소계",
        "일반용1","일반용2","일반용소계",
        "업무난방용","냉난방용","업무용소계",
        "산업용","열병합용1","열병합용2","열전용설비용","수송용","연료전지용",
        "합계","증감률","구성비",
    ]  # 18

    if year <= 2004:
        base = labels_2001_2004
    elif year <= 2018:
        base = labels_2005_2018
    else:
        base = labels_2019_plus

    if width == len(base):
        return base
    elif width < len(base):
        return base[:width]
    else:
        extra = [f"col{i+1}" for i in range(width - len(base))]
        return base + extra

# ================== 수요가수 블록 정리 ==================
def finalize_demand_block(df_block: pd.DataFrame, year: int) -> pd.DataFrame:
    # 파생
    s_g  = as_series(df_block, "가정용") if "가정용" in df_block.columns else 0
    s_nb = as_series(df_block, "난방용") if "난방용" in df_block.columns else 0
    s_up = as_series(df_block, "업무용") if "업무용" in df_block.columns else 0
    s_nn = as_series(df_block, "냉난방용") if "냉난방용" in df_block.columns else 0
    set_series(df_block, "취사용",     s_g.fillna(0) - s_nb.fillna(0))
    set_series(df_block, "업무난방용", s_up.fillna(0) - s_nn.fillna(0))
    set_series(df_block, "개별난방용", s_nb if isinstance(s_nb, pd.Series) else pd.NA)

    # 2019+: 열병합용1+2 → 열병합용, 열전용설비용 → 집단에너지용
    if year >= 2019:
        if "열병합용1" in df_block.columns or "열병합용2" in df_block.columns:
            a = df_block.get("열병합용1", 0)
            b = df_block.get("열병합용2", 0)
            set_series(
                df_block,
                "열병합용",
                pd.to_numeric(a, errors="coerce").fillna(0)
                + pd.to_numeric(b, errors="coerce").fillna(0),
            )
        if "열전용설비용" in df_block.columns:
            df_block = df_block.rename(columns={"열전용설비용": "집단에너지용"})

    # 2005~2018: 집단용 → 집단에너지용 (연속성)
    if "집단용" in df_block.columns and "집단에너지용" not in df_block.columns:
        df_block = df_block.rename(columns={"집단용": "집단에너지용"})

    # 공통 제거(계/합계/증감률/구성비)
    drop_cols = [c for c in df_block.columns if ("계" in str(c))] + ["증감률","구성비"]
    drop_cols = [c for c in set(drop_cols) if c not in ("시도","회사")]
    df_block = df_block.drop(columns=drop_cols, errors="ignore")

    # 노이즈 정리
    df_block = drop_noise_rows(df_block)
    df_block = drop_mostly_text_cols(df_block)

    # 최종 보존
    keep = [
        "시도","회사",
        "취사용","개별난방용","일반용1","일반용2",
        "업무난방용","냉난방용","산업용",
        "열병합용","집단에너지용","연료전지용","수송용",
    ]
    for c in keep:
        if c not in df_block.columns:
            df_block[c] = pd.NA
    return df_block[keep].copy()

# ================== 공급량 블록(소계 보정 포함) ==================
def _find_sum_col_index(arr: np.ndarray, tol_ratio: float = 0.02) -> int | None:
    if arr.shape[1] != 3:
        return None
    candidates = []
    for j in range(3):
        others = [k for k in range(3) if k != j]
        sum_others = (arr[:, others[0]] + arr[:, others[1]])
        denom = np.maximum(1, np.abs(sum_others))
        ok = np.isfinite(arr[:, j]) & np.isfinite(sum_others)
        ratio_ok = np.zeros_like(ok, dtype=bool)
        ratio_ok[ok] = np.abs(arr[ok, j] - sum_others[ok]) / denom[ok] < tol_ratio
        candidates.append((ratio_ok.mean(), j))
    candidates.sort(reverse=True)
    return candidates[0][1] if candidates[0][0] >= 0.5 else None

def normalize_triplet(df: pd.DataFrame, cols_in: list[str], cols_out: list[str]) -> pd.DataFrame:
    A = df[cols_in].to_numpy(dtype=float)
    sum_idx = _find_sum_col_index(A)
    if sum_idx is None:
        return df.rename(columns=dict(zip(cols_in, cols_out)))
    order = [i for i in range(3) if i != sum_idx] + [sum_idx]
    return df.rename(columns={
        cols_in[order[0]]: cols_out[0],
        cols_in[order[1]]: cols_out[1],
        cols_in[order[2]]: cols_out[2],
    })

def finalize_supply_block(df_block: pd.DataFrame, year: int) -> pd.DataFrame:
    width = df_block.shape[1] - 2
    df_block.columns = ["시도","회사"] + supply_labels(year, width)

    clean_fill_idcols(df_block)
    num_cols = [c for c in df_block.columns if c not in ("시도","회사")]
    numify(df_block, num_cols)

    # 가정용/일반용/업무용 3열 세트 소계 위치 보정
    cand_home = [c for c in df_block.columns if c in ["취사용","개별난방용","가정용소계","취사","난방","소계"]]
    if len(cand_home) >= 3:
        picks = []
        for nm in ["취사용","개별난방용","가정용소계","취사","난방","소계"]:
            if nm in df_block.columns and nm not in picks:
                picks.append(nm)
            if len(picks) == 3:
                break
        df_block = normalize_triplet(df_block, picks, ["취사용","개별난방용","가정용소계"])

    cand_gen = [c for c in df_block.columns if c in ["일반용1","일반용2","일반용소계","영업1","영업2","소계"]]
    if len(cand_gen) >= 3:
        picks = []
        for nm in ["일반용1","일반용2","일반용소계","영업1","영업2","소계"]:
            if nm in df_block.columns and nm not in picks:
                picks.append(nm)
            if len(picks) == 3:
                break
        df_block = normalize_triplet(df_block, picks, ["일반용1","일반용2","일반용소계"])

    cand_off = [c for c in df_block.columns if c in ["업무난방용","냉난방용","업무용소계","난방","냉방","소계"]]
    if len(cand_off) >= 3:
        picks = []
        for nm in ["업무난방용","냉난방용","업무용소계","난방","냉방","소계"]:
            if nm in df_block.columns and nm not in picks:
                picks.append(nm)
            if len(picks) == 3:
                break
        df_block = normalize_triplet(df_block, picks, ["업무난방용","냉난방용","업무용소계"])

    # 2019+: 열병합용1+2 → 열병합용, 열전용설비용 → 집단에너지용
    if year >= 2019:
        if "열병합용1" in df_block.columns or "열병합용2" in df_block.columns:
            a = df_block.get("열병합용1", 0)
            b = df_block.get("열병합용2", 0)
            set_series(
                df_block,
                "열병합용",
                pd.to_numeric(a, errors="coerce").fillna(0)
                + pd.to_numeric(b, errors="coerce").fillna(0),
            )
        if "열전용설비용" in df_block.columns:
            df_block = df_block.rename(columns={"열전용설비용": "집단에너지용"})

    # 2005~2018: 집단용 → 집단에너지용
    if "집단용" in df_block.columns and "집단에너지용" not in df_block.columns:
        df_block = df_block.rename(columns={"집단용": "집단에너지용"})

    # 소계/합계/증감률/구성비 제거
    df_block = drop_noise_rows(df_block)
    df_block = drop_mostly_text_cols(df_block)
    drop_cols = [
        c for c in df_block.columns
        if ("소계" in str(c)) or (c in {"합계","증감률","구성비"})
    ]
    df_block = df_block.drop(columns=drop_cols, errors="ignore")

    keep = [
        "시도","회사",
        "취사용","개별난방용","일반용1","일반용2",
        "업무난방용","냉난방용","산업용",
        "열병합용","집단에너지용","연료전지용","수송용",
    ]
    for c in keep:
        if c not in df_block.columns:
            df_block[c] = pd.NA
    return df_block[keep].copy()

# ================== 시트 처리 ==================
def process_sheet_like(df_raw: pd.DataFrame, year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    siho = df_raw.iloc[:, :2].copy()
    rest = df_raw.iloc[:, 2:].copy()

    d_w, s_w = get_block_widths(year)
    need = d_w + s_w
    if rest.shape[1] < need:
        raise ValueError(
            f"{year}년: 실제 나머지 열 {rest.shape[1]} < 기대 {need} (2 + {d_w} + {s_w})"
        )
    rest = rest.iloc[:, :need]

    left  = rest.iloc[:, :d_w].copy()
    right = rest.iloc[:, d_w:d_w+s_w].copy()

    # 수요
    demand_block = pd.concat([siho, left], axis=1)
    demand_block.columns = ["시도","회사"] + make_demand_labels(d_w, year)
    clean_fill_idcols(demand_block)
    num_cols = [c for c in demand_block.columns if c not in ("시도","회사")]
    numify(demand_block, num_cols)
    demand_block = finalize_demand_block(demand_block, year)

    # 공급
    supply_block = pd.concat([siho, right], axis=1)
    supply_block.columns = ["시도","회사"] + supply_labels(year, s_w)
    clean_fill_idcols(supply_block)
    num_cols_s = [c for c in supply_block.columns if c not in ("시도","회사")]
    numify(supply_block, num_cols_s)
    supply_block = finalize_supply_block(supply_block, year)

    d_long = demand_block.melt(
        id_vars=["시도","회사"], var_name="용도", value_name="수요가수"
    )
    s_long = supply_block.melt(
        id_vars=["시도","회사"], var_name="용도", value_name="공급량"
    )
    d_long.insert(0, "연도", year)
    s_long.insert(0, "연도", year)
    return d_long, s_long

def sum_keep_nan(x: pd.Series):
    return x.sum(min_count=1)

def dedup_and_merge(d_long: pd.DataFrame, s_long: pd.DataFrame) -> pd.DataFrame:
    keys = ["연도","시도","회사","용도"]
    d_long = d_long.groupby(keys, as_index=False, dropna=False).agg({"수요가수": sum_keep_nan})
    s_long = s_long.groupby(keys, as_index=False, dropna=False).agg({"공급량": sum_keep_nan})
    return pd.merge(d_long, s_long, on=keys, how="outer", validate="one_to_one")

# ================== 실행: 통합 + 연도별 파일 ==================
rows_all = []

# 1) 통합(2001~2015)
for f in sorted(DATA.glob("*2001_2015*.xlsx")):
    tmp = TMP_DIR / f"{f.stem}_tmp_unmerged.xlsx"
    unmerge_save(f, tmp)
    xf = pd.ExcelFile(tmp)
    for sn in xf.sheet_names:
        m = re.search(r"(\d{4})", sn)
        if not m:
            continue
        year = int(m.group(1))
        raw = pd.read_excel(tmp, sheet_name=sn, header=None, dtype=str)
        df  = raw.iloc[5:].reset_index(drop=True)
        if df.shape[1] < 6:
            continue
        d_long, s_long = process_sheet_like(df, year)
        rows_all.append(dedup_and_merge(d_long, s_long))

# 2) 연도별 파일(2016~) — '부피' 시트만 처리
pat = r".*\((\d{4})\)\.xlsx$"
for f in sorted(DATA.glob("*.xlsx")):
    name_lower = f.name.lower()
    if "2001_2015" in name_lower or name_lower.endswith("_tmp_unmerged.xlsx"):
        continue
    m = re.search(pat, f.name)
    if not m:
        continue
    year = int(m.group(1))
    tmp = TMP_DIR / f"{f.stem}_tmp_unmerged.xlsx"
    unmerge_save(f, tmp)

    xf = pd.ExcelFile(tmp)
    sheet_vol = pick_volume_sheet(xf)   # '부피' 시트 탐색
    raw = pd.read_excel(tmp, sheet_name=sheet_vol, header=None, dtype=str)
    df  = raw.iloc[5:].reset_index(drop=True)
    if df.shape[1] < 6:
        continue
    d_long, s_long = process_sheet_like(df, year)
    rows_all.append(dedup_and_merge(d_long, s_long))

# 3) 저장
final_df = (
    pd.concat(rows_all, ignore_index=True)
    if rows_all else
    pd.DataFrame(columns=["연도","시도","회사","용도","수요가수","공급량"])
)

out_path = OUT / "용도별_수요가수_공급량_(2001-현재).csv"
final_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("저장 완료:", out_path)
print("행 수:", len(final_df))
print("컬럼:", list(final_df.columns))

# (옵션) Temp 정리
# for p in TMP_DIR.glob("*_tmp_unmerged.xlsx"):
#     p.unlink(missing_ok=True)
